In [ ]:
import pandas as pd
import json
import random

data = pd.read_parquet('imdb.parquet')


In [ ]:
target_per_label = 500
positive_samples = []
negative_samples = []

for _, row in data.iterrows():
    text = row['text']
    if len(text) > 500 or len(text) < 200:
        continue

    cleaned_text = ' '.join(text.replace('<br />', ' ').split())
    entry = {
        'text': cleaned_text,
        'answer': 'Positive' if row['label'] == 1 else 'Negative',
        'label': int(row['label'])
    }

    if row['label'] == 1 and len(positive_samples) < target_per_label:
        positive_samples.append(entry)
    elif row['label'] == 0 and len(negative_samples) < target_per_label:
        negative_samples.append(entry)

    if len(positive_samples) == target_per_label and len(negative_samples) == target_per_label:
        break

if len(positive_samples) < target_per_label or len(negative_samples) < target_per_label:
    raise ValueError("Unable to collect balanced samples with current constraints.")

json_data = positive_samples + negative_samples
random.shuffle(json_data)

for idx, sample in enumerate(json_data):
    sample['id'] = idx

positive_count = sum(sample['label'] == 1 for sample in json_data)
negative_count = sum(sample['label'] == 0 for sample in json_data)
print(f"Positive samples: {positive_count}")
print(f"Negative samples: {negative_count}")

with open('../imdb.json', 'w') as f:
    json.dump(json_data, f, indent=2)